> ## 📦 ARCHIVED — 600k 历史记录 (2026-05-07)
>
> **本 notebook 保留作 600k cross_u10 historical artefact**(eval_log + final_eval +
> trainer_state 已落 `experiments/arrival_v2_prototype/cross_u10_regression/...`),不要重跑。
>
> **600k 结论**:`final=0.867 (PASS)`、`last100k_mean=0.6533 (FAIL gate 0.78)`、
> `OOB=0.033 (PASS)`、`READY_FOR_P1_V6=False`。失败模式判定为「未收敛」非「奖励学坏」
> (eval 曲线 525k→600k:0.467→0.633→0.700→0.867,`path_efficiency` 0.57→0.75 仍在涨)。
>
> **续训 + P1 v6 在新 notebook**:[`sac_arrival_v2_cross_extension_and_p1_v6.ipynb`](./sac_arrival_v2_cross_extension_and_p1_v6.ipynb)
> (cross 续到 1M + P1 v6 1.5M,改用 `!python -u` 实时输出)。

---

# SAC arrival_v2 — cross_u10 Behavior Regression

**目的**：先用 A0 known-good 的 `single_u10_cross_tgt15` 做 600k behavior regression；若通过，再接着跑真正暴露旧问题的 P1 v6 (`s1 × u15_upstream × 1M`)。

| 项 | 值 |
|---|---|
| benchmark | `single_u10_cross_tgt15` |
| flow | `wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy` |
| algo | vanilla SAC（无 LayerNorm / 无 AsymCritic / UTD=1） |
| reward | `arrival_v2` (`w_safety=2.0`, `R_fast_success=0`) |
| sensor | `s0_k4` |
| seed | `46` |
| total steps | `600000` |
| eval | `benchmarks/single_u10_cross_tgt15.json`，30 episodes |

**通过判据**：final success ≥ 0.85；last100k mean success ≥ 0.9 × peak success；final OOB rate ≤ 0.10。

**输出位置**：
- cross gate: `experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46/`
- P1 v6: `experiments/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46/`


## 0. GPU sanity

In [ ]:
!nvidia-smi | head -10
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")


Thu May  7 11:05:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
torch=2.10.0+cu128 cuda=True device=NVIDIA L4


## 1. Mount Drive + cwd

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Keep this path aligned with the working clone that contains commit 813096e
# plus the doc correction in docs/online_sac_reward_redesign.md.
REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. 通用配置

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

BENCHMARK_KEY = 'single_u10_cross_tgt15'
FLOW_PATH = 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
TASK_GEOMETRY = 'cross_stream'
TARGET_SPEED = 1.5
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 4
SEED = 46

TOTAL_STEPS = 600_000
RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

RUN_ROOT = Path('experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46')
CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46')
MANIFEST_PATH = Path(f'benchmarks/{BENCHMARK_KEY}.json')

SHELL_ENV = {
    'BENCHMARK_KEY': BENCHMARK_KEY,
    'FLOW_PATH': FLOW_PATH,
    'TASK_GEOMETRY': TASK_GEOMETRY,
    'TARGET_SPEED': TARGET_SPEED,
    'OBJECTIVE': OBJECTIVE,
    'PROBE_LAYOUT': PROBE_LAYOUT,
    'HISTORY_LENGTH': HISTORY_LENGTH,
    'SEED': SEED,
    'TOTAL_STEPS': TOTAL_STEPS,
    'RANDOM_STEPS': RANDOM_STEPS,
    'UPDATE_AFTER': UPDATE_AFTER,
    'BATCH_SIZE': BATCH_SIZE,
    'HIDDEN_DIM': HIDDEN_DIM,
    'NUM_ENVS': NUM_ENVS,
    'EVAL_EVERY': EVAL_EVERY,
    'EVAL_EPISODES': EVAL_EPISODES,
    'CHECKPOINT_EVERY': CHECKPOINT_EVERY,
    'DEVICE': DEVICE,
    'RUN_ROOT': str(RUN_ROOT),
    'CKPT_ROOT': str(CKPT_ROOT),
    'MANIFEST_PATH': str(MANIFEST_PATH),
}
os.environ.update({key: str(value) for key, value in SHELL_ENV.items()})
os.environ['PYTHONUNBUFFERED'] = '1'

print(f'benchmark       = {BENCHMARK_KEY}')
print(f'flow            = {FLOW_PATH}')
print(f'objective       = {OBJECTIVE}')
print(f'run_root        = {RUN_ROOT}')
print(f'checkpoint_root = {CKPT_ROOT}')
print(f'total_steps     = {TOTAL_STEPS:,}')


benchmark       = single_u10_cross_tgt15
flow            = wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy
objective       = arrival_v2
run_root        = experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46
checkpoint_root = checkpoints/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46
total_steps     = 600,000


## 3. Preflight — repo / flow / gates

停止条件：flow 缺失、manifest 生成失败、validator 失败、reward unit tests 失败。

In [ ]:
flow_path = Path(FLOW_PATH)
if not flow_path.exists():
    raise FileNotFoundError(f'missing flow file: {flow_path}')
print(f'[OK] flow exists: {flow_path} ({flow_path.stat().st_size / 1e6:.1f} MB)')


[OK] flow exists: wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy (230.4 MB)


In [ ]:
!python -m scripts.validate_arrival_v2_candidate
!python -m pytest tests/test_reward_objective.py -q
!python -m scripts.generate_standard_benchmarks --benchmarks {BENCHMARK_KEY} --episodes {EVAL_EPISODES}

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {MANIFEST_PATH}')
print(f'[OK] manifest ready: {MANIFEST_PATH}')



[undiscounted]
fast_success       144.473
slow_success       142.023
unsafe_success      91.048
timeout_near      -149.800
timeout_far       -229.800
late_oob          -274.500
fast_oob          -280.325
mid_oob           -319.450

[discounted_gamma_0.995]
fast_success        88.578
slow_success        54.683
unsafe_success      34.857
timeout_near       -39.480
timeout_far        -58.269
late_oob          -103.800
mid_oob           -190.006
fast_oob          -212.555

[discounted_shortcut] safe=69.330 risky=62.361

PASS: arrival_v2 candidate pre-integration gates passed.
...                                                                      [100%]
3 passed in 18.50s
[done] single_u10_cross_tgt15 -> benchmarks/single_u10_cross_tgt15.json
[OK] manifest ready: benchmarks/single_u10_cross_tgt15.json


## 4. 跑 cross_u10 behavior regression

如果 `final_eval.json` 已存在，本 cell 会跳过训练，避免误覆盖已完成结果。

In [ ]:
%%bash
set -euo pipefail

FINAL_EVAL="${RUN_ROOT}/results/final_eval.json"
echo "[cmd] python -u -m scripts.train_sac --objective ${OBJECTIVE} --flow ${FLOW_PATH} --eval-manifest ${MANIFEST_PATH}"
if [[ -f "${FINAL_EVAL}" ]]; then
  echo "[skip] final_eval already exists: ${FINAL_EVAL}"
else
  time python -u -m scripts.train_sac     --flow "${FLOW_PATH}"     --task-geometry "${TASK_GEOMETRY}"     --target-speed "${TARGET_SPEED}"     --objective "${OBJECTIVE}"     --probe-layout "${PROBE_LAYOUT}"     --history-length "${HISTORY_LENGTH}"     --total-steps "${TOTAL_STEPS}"     --random-steps "${RANDOM_STEPS}"     --update-after "${UPDATE_AFTER}"     --batch-size "${BATCH_SIZE}"     --hidden-dim "${HIDDEN_DIM}"     --num-envs "${NUM_ENVS}"     --eval-every "${EVAL_EVERY}"     --eval-episodes "${EVAL_EPISODES}"     --checkpoint-every "${CHECKPOINT_EVERY}"     --eval-manifest "${MANIFEST_PATH}"     --seed "${SEED}"     --device "${DEVICE}"     --save-dir "${RUN_ROOT}"     --checkpoint-dir "${CKPT_ROOT}"
fi


[cmd] python -u -m scripts.train_sac --objective arrival_v2 --flow wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy --eval-manifest benchmarks/single_u10_cross_tgt15.json
[train] episode=5 step=774 return=-305.55 success=False time=64.5s geometry=cross_stream history=4
[train] episode=10 step=1314 return=-273.05 success=False time=13.3s geometry=cross_stream history=4
[train] episode=15 step=2124 return=-304.23 success=False time=42.9s geometry=cross_stream history=4
[train] episode=20 step=2742 return=-402.36 success=False time=102.4s geometry=cross_stream history=4
[train] episode=25 step=3612 return=-309.07 success=False time=71.6s geometry=cross_stream history=4
[train] episode=30 step=4524 return=-349.71 success=False time=75.3s geometry=cross_stream history=4
[train] episode=35 step=5298 return=-340.20 success=False time=79.8s geometry=cross_stream history=4 | q1=1.366 actor=1.242 alpha=0.197
[train] episode=40 step=6132 return=-288.96 success=False time=52.6s geo


real	71m6.778s
user	160m35.562s
sys	0m51.399s


## 5. Summarize — eval curve + final termination

In [ ]:
eval_log_path = RUN_ROOT / 'results' / 'eval_log.csv'
final_eval_path = RUN_ROOT / 'results' / 'final_eval.json'
trainer_state_path = RUN_ROOT / 'trainer_state.json'

if not eval_log_path.exists():
    raise FileNotFoundError(f'missing eval log: {eval_log_path}')
if not final_eval_path.exists():
    raise FileNotFoundError(f'missing final eval: {final_eval_path}')

df = pd.read_csv(eval_log_path)
final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
last100 = df[df['env_step'] >= TOTAL_STEPS - 100_000].copy()
last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
final_success = float(final_eval['eval_success_rate'])
counts = final_eval.get('eval_termination_counts', {})
num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

print('=' * 96)
print(f"{'metric':<32}{'value':>18}{'gate':>18}")
print('-' * 96)
print(f"{'final_success_rate':<32}{final_success:>18.4f}{PASS_FINAL_SUCCESS:>18.4f}")
print(f"{'peak_success_rate':<32}{peak_success:>18.4f}{'':>18}")
print(f"{'last100k_mean_success':<32}{last100_mean:>18.4f}{PASS_LAST100_RATIO * peak_success:>18.4f}")
print(f"{'final_oob_rate':<32}{oob_rate:>18.4f}{PASS_OOB_RATE:>18.4f}")
print(f"{'obs_dim':<32}{trainer_state.get('observation_dim', 'NA'):>18}{'':>18}")
print(f"{'episode_context_obs':<32}{str(trainer_state.get('include_episode_context_obs', 'NA')):>18}{'expected True':>18}")
print(f"{'timeout_bootstrap':<32}{str(trainer_state.get('timeout_bootstrap_semantics', 'NA')):>18}{'expected terminal':>18}")
print('=' * 96)
print('termination_counts:', counts)

print()
print('[last eval rows]')
cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost', 'eval_time_s', 'eval_progress_ratio']
print(df[cols].tail(8).to_string(index=False))


metric                                       value              gate
------------------------------------------------------------------------------------------------
final_success_rate                          0.8667            0.8500
peak_success_rate                           0.8667                  
last100k_mean_success                       0.6533            0.7800
final_oob_rate                              0.0333            0.1000
obs_dim                                         48                  
episode_context_obs                           True     expected True
timeout_bootstrap                         terminal expected terminal
termination_counts: {'goal': 26, 'timeout': 3, 'out_of_bounds': 1}

[last eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   425004           0.466667   -10.599755         10.439441   118.803333             0.696786
   450000           0.366667   -60.976655         18.936237   108.350000      

## 6. Verdict — 是否允许进入 P1 v6

In [ ]:
checks = [
    ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
    ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success, f'{last100_mean:.4f} / peak={peak_success:.4f}'),
    ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
    ('arrival_v2 context obs enabled', trainer_state.get('include_episode_context_obs') is True, str(trainer_state.get('include_episode_context_obs'))),
    ('arrival_v2 timeout terminal semantics', trainer_state.get('timeout_bootstrap_semantics') == 'terminal', str(trainer_state.get('timeout_bootstrap_semantics'))),
]

print('=' * 88)
print(f"{'check':<48}{'pass':>8}{'detail':>28}")
print('-' * 88)
all_pass = True
for name, ok, detail in checks:
    mark = 'PASS' if ok else 'FAIL'
    if not ok:
        all_pass = False
    print(f'{name:<48}{mark:>8}{detail:>28}')
print('=' * 88)
print()
print('READY_FOR_P1_V6=' + str(all_pass))

summary = {
    'benchmark': BENCHMARK_KEY,
    'objective': OBJECTIVE,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'seed': SEED,
    'total_steps': TOTAL_STEPS,
    'final_success_rate': final_success,
    'peak_success_rate': peak_success,
    'last100k_mean_success': last100_mean,
    'final_oob_rate': oob_rate,
    'termination_counts': counts,
    'ready_for_p1_v6': all_pass,
    'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
}
summary_path = RUN_ROOT / 'results' / 'cross_u10_gate_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'[saved] {summary_path}')


check                                               pass                      detail
----------------------------------------------------------------------------------------
final success >= 0.85                               PASS                      0.8667
last100k mean >= 0.9 * peak                         FAIL        0.6533 / peak=0.8667
final OOB rate <= 0.10                              PASS                      0.0333
arrival_v2 context obs enabled                      PASS                        True
arrival_v2 timeout terminal semantics               PASS                    terminal

READY_FOR_P1_V6=False
[saved] experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46/results/cross_u10_gate_summary.json


## 7. P1 v6 — upstream 主实验（cross gate 通过后自动跑）

这里才是真正验证 `arrival_v2` 是否修复旧 failure mode 的实验：`single_u15_upstream_tgt15 / s1_k4 / seed 46 / 1M steps`。

默认逻辑：`RUN_P1_V6 = all_pass`。如果 cross gate 没过，本 cell 会停止；如果已过，会直接继续跑 P1 v6。


In [ ]:
RUN_P1_V6 = bool(all_pass)

P1_BENCHMARK_KEY = 'single_u15_upstream_tgt15'
P1_FLOW_PATH = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
P1_MANIFEST_PATH = Path(f'benchmarks/{P1_BENCHMARK_KEY}.json')
P1_RUN_ROOT = Path('experiments/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46')
P1_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46')
P1_TOTAL_STEPS = 1_000_000

P1_SHELL_ENV = {
    'RUN_P1_V6': str(RUN_P1_V6),
    'P1_BENCHMARK_KEY': P1_BENCHMARK_KEY,
    'P1_FLOW_PATH': P1_FLOW_PATH,
    'P1_MANIFEST_PATH': str(P1_MANIFEST_PATH),
    'P1_RUN_ROOT': str(P1_RUN_ROOT),
    'P1_CKPT_ROOT': str(P1_CKPT_ROOT),
    'P1_TOTAL_STEPS': P1_TOTAL_STEPS,
}
os.environ.update({key: str(value) for key, value in P1_SHELL_ENV.items()})

print('READY_FOR_P1_V6=' + str(RUN_P1_V6))
print(f'p1_benchmark    = {P1_BENCHMARK_KEY}')
print(f'p1_flow         = {P1_FLOW_PATH}')
print(f'p1_run_root     = {P1_RUN_ROOT}')
print(f'p1_total_steps  = {P1_TOTAL_STEPS:,}')


READY_FOR_P1_V6=False
p1_benchmark    = single_u15_upstream_tgt15
p1_flow         = wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy
p1_run_root     = experiments/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46
p1_total_steps  = 1,000,000


In [ ]:
%%bash
set -euo pipefail

echo "READY_FOR_P1_V6=${RUN_P1_V6}"
if [[ "${RUN_P1_V6}" != "True" ]]; then
  echo "[hold] cross_u10 gate did not pass; hold P1 v6"
  exit 1
fi

P1_FINAL_EVAL="${P1_RUN_ROOT}/results/final_eval.json"
echo "[cmd] python -u -m scripts.train_sac --objective ${OBJECTIVE} --flow ${P1_FLOW_PATH} --eval-manifest ${P1_MANIFEST_PATH}"
if [[ -f "${P1_FINAL_EVAL}" ]]; then
  echo "[skip] P1 final_eval already exists: ${P1_FINAL_EVAL}"
else
  if [[ ! -f "${P1_FLOW_PATH}" ]]; then
    echo "[FAIL] missing P1 flow file: ${P1_FLOW_PATH}" >&2
    exit 1
  fi
  python -u -m scripts.generate_standard_benchmarks     --benchmarks "${P1_BENCHMARK_KEY}"     --episodes "${EVAL_EPISODES}"

  time python -u -m scripts.train_sac     --flow "${P1_FLOW_PATH}"     --task-geometry upstream     --target-speed "${TARGET_SPEED}"     --objective "${OBJECTIVE}"     --probe-layout s1     --history-length "${HISTORY_LENGTH}"     --total-steps "${P1_TOTAL_STEPS}"     --random-steps "${RANDOM_STEPS}"     --update-after "${UPDATE_AFTER}"     --batch-size "${BATCH_SIZE}"     --hidden-dim "${HIDDEN_DIM}"     --num-envs "${NUM_ENVS}"     --eval-every "${EVAL_EVERY}"     --eval-episodes "${EVAL_EPISODES}"     --checkpoint-every "${CHECKPOINT_EVERY}"     --eval-manifest "${P1_MANIFEST_PATH}"     --seed "${SEED}"     --device "${DEVICE}"     --save-dir "${P1_RUN_ROOT}"     --checkpoint-dir "${P1_CKPT_ROOT}"
fi


READY_FOR_P1_V6=False
[hold] cross_u10 gate did not pass; hold P1 v6


CalledProcessError: Command 'b'set -euo pipefail\n\necho "READY_FOR_P1_V6=${RUN_P1_V6}"\nif [[ "${RUN_P1_V6}" != "True" ]]; then\n  echo "[hold] cross_u10 gate did not pass; hold P1 v6"\n  exit 1\nfi\n\nP1_FINAL_EVAL="${P1_RUN_ROOT}/results/final_eval.json"\necho "[cmd] python -u -m scripts.train_sac --objective ${OBJECTIVE} --flow ${P1_FLOW_PATH} --eval-manifest ${P1_MANIFEST_PATH}"\nif [[ -f "${P1_FINAL_EVAL}" ]]; then\n  echo "[skip] P1 final_eval already exists: ${P1_FINAL_EVAL}"\nelse\n  if [[ ! -f "${P1_FLOW_PATH}" ]]; then\n    echo "[FAIL] missing P1 flow file: ${P1_FLOW_PATH}" >&2\n    exit 1\n  fi\n  python -u -m scripts.generate_standard_benchmarks     --benchmarks "${P1_BENCHMARK_KEY}"     --episodes "${EVAL_EPISODES}"\n\n  time python -u -m scripts.train_sac     --flow "${P1_FLOW_PATH}"     --task-geometry upstream     --target-speed "${TARGET_SPEED}"     --objective "${OBJECTIVE}"     --probe-layout s1     --history-length "${HISTORY_LENGTH}"     --total-steps "${P1_TOTAL

In [ ]:
p1_final_eval_path = P1_RUN_ROOT / 'results' / 'final_eval.json'
if not p1_final_eval_path.exists():
    raise FileNotFoundError(f'missing P1 final eval: {p1_final_eval_path}')

p1_final = json.loads(p1_final_eval_path.read_text(encoding='utf-8'))
p1_counts = p1_final.get('eval_termination_counts', {})
print('=' * 88)
print('P1_V6_FINAL')
print(f"success_rate     : {p1_final['eval_success_rate']:.4f}")
print(f"avg_return       : {p1_final['eval_return']:.2f} +/- {p1_final['eval_return_std']:.2f}")
print(f"avg_time_s       : {p1_final['eval_time_s']:.2f}")
print(f"progress_ratio   : {p1_final['eval_progress_ratio']:.4f}")
print(f"safety_cost      : {p1_final['eval_safety_cost']:.4f}")
print(f"termination      : {p1_counts}")
print('=' * 88)
